

Una feature derivada es una **columna nueva** que el modelo de ML no tendria
naturalmente, calculada a partir de las columnas originales. 
La idea es darle

al modelo informacion **pre-masticada** que captura un patron que el modelo
no descubriria por si solo.

Ejemplo simple: si tengo `ingreso` y `monto_prestamo`, el modelo lineal los ve
como dos numeros independientes. Pero la **relacion entre ellos** (cuanto del
ingreso compromete el prestamo) es lo que realmente importa para predecir
default. Esa relacion es una feature derivada.

## ¿Que significa una correlacion de 0.46?

La correlacion mide **que tan relacionadas estan dos variables**. Va de -1 a +1:

- **+1.0** = relacion perfecta directa (sube una, sube la otra)
- **0.0** = sin relacion (son independientes)
- **-1.0** = relacion perfecta inversa (sube una, baja la otra)

Reglas de pulgar para predecir un target binario (default si/no):

| \|corr\| | Interpretacion |
|---|---|
| < 0.10 | Sin señal util |
| 0.10 - 0.20 | Señal debil |
| 0.20 - 0.40 | Señal moderada |
| 0.40 - 0.60 | Señal fuerte |
| > 0.60 | Señal muy fuerte (raro en problemas reales) |

Nuestras 3 features estan entre 0.40 y 0.54: **señal fuerte y solida**.

---

## Setup

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'loan_data.csv')
df.shape

(45000, 14)

---
## Feature 1: `has_prev_defaults`

### ¿Que es?

Un **encoding binario** de la columna categorica `previous_loan_defaults_on_file`:
- Si vale `"Yes"` → 1
- Si vale `"No"` → 0

### ¿Por que necesita encoding?

Los modelos de ML trabajan con numeros, no con texto. La cadena `"Yes"` no significa
nada para un modelo — hay que **traducirla** a un numero. La forma mas simple para
una variable binaria es 1/0.

### ¿Que captura?

El **historial crediticio** del solicitante. La intuicion bancaria clasica dice:
*"si alguien ya entro en default antes, es mas probable que lo vuelva a hacer"*.

Pero en **este dataset particular**, la relacion es **inversa**: los que tienen
defaults previos defaultean **menos**. Probablemente porque:
- Los que ya defaultearon y siguieron en el sistema = aprendieron la leccion.
- O el dataset tiene un sesgo de seleccion (se aprobaron solo aquellos que pagaron sus defaults).

Sea como sea, **la señal es real**: la columna discrimina mas que cualquier otra
del dataset. El modelo la usa, no importa la direccion.

In [2]:
df['has_prev_defaults'] = (df['previous_loan_defaults_on_file'] == 'Yes').astype(int)

# Distribucion
print(df['has_prev_defaults'].value_counts())
print()

# Tasa de default por valor
print(df.groupby('has_prev_defaults').agg(
    n=('loan_status', 'size'),
    tasa_default=('loan_status', 'mean'),
).round(4))

has_prev_defaults
1    22858
0    22142
Name: count, dtype: int64

                       n  tasa_default
has_prev_defaults                     
0                  22142        0.4516
1                  22858        0.0000


In [3]:
corr = df['has_prev_defaults'].corr(df['loan_status'])
print(f'correlacion con loan_status: {corr:.4f}  (|corr|={abs(corr):.4f})')

correlacion con loan_status: -0.5431  (|corr|=0.5431)


### Lectura

Los 22.142 sin defaults previos defaultean en 45%. Los 22.858 con defaults previos
defaultean en 0%. **Spread total: 45 puntos porcentuales**. La correlacion sale
**-0.54**: negativa (Yes asocia a 0 default) y fuerte.

Es el predictor mas potente del dataset entero.

---
## Feature 2: `loan_burden`

### ¿Que es?

Una **ratio financiero**: el costo total del prestamo (incluyendo intereses)
como fraccion del ingreso anual del solicitante.

### Formula

```
loan_burden = loan_amnt × (1 + loan_int_rate/100) / person_income
```

### Traduccion a lenguaje natural

> "Si te aprueban un prestamo de $10.000 al 12% anual y ganas $40.000 al año,
> el costo total es $10.000 × 1.12 = $11.200, lo que equivale a 0.28 de tu
> ingreso anual. `loan_burden = 0.28`."

### ¿Que captura?

La **capacidad de pago**. No es lo mismo deberle $10.000 al banco si ganas $1M
(loan_burden = 0.01, despreciable) que si ganas $30.000 (loan_burden = 0.37,
una carga seria).

Esta feature responde la pregunta:
> *"¿Cuanto del ingreso del cliente compromete el costo total del prestamo?"*

Cuanto mas alto el valor, mayor el riesgo de default.

In [4]:
df['loan_burden'] = (df['loan_amnt'] * (1 + df['loan_int_rate'] / 100)) / df['person_income'].clip(lower=1)

# Distribucion
df['loan_burden'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(4)

count    45000.0000
mean         0.1555
std          0.0976
min          0.0007
25%          0.0812
50%          0.1348
75%          0.2095
90%          0.2939
95%          0.3459
max          0.7290
Name: loan_burden, dtype: float64

In [5]:
# Tasa de default por quintiles de loan_burden
df['burden_quintil'] = pd.qcut(df['loan_burden'], q=5, labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)'])
df.groupby('burden_quintil', observed=True).agg(
    n=('loan_status', 'size'),
    tasa_default=('loan_status', 'mean'),
).round(4)

,n,tasa_default
burden_quintil,,
Q1 (bajo),9000,0.1017
Q2,9000,0.1252
Q3,9000,0.1524
Q4,9000,0.1964
Q5 (alto),9000,0.5353


In [6]:
corr = df['loan_burden'].corr(df['loan_status'])
print(f'correlacion con loan_status: {corr:.4f}  (|corr|={abs(corr):.4f})')

correlacion con loan_status: 0.3972  (|corr|=0.3972)


### Lectura

El quintil con menor `loan_burden` (Q1) defaultea ~10%. El quintil mas alto (Q5)
defaultea ~53%. **Multiplica el riesgo por 5x**. Correlacion 0.40, señal fuerte.

---
## Feature 3: `rate_x_pct_income`

### ¿Que es?

Una **interaccion**: el producto de dos columnas originales que, **multiplicadas**,
revelan un patron que ninguna captura sola.

### Formula

```
rate_x_pct_income = loan_int_rate × loan_percent_income
```

### ¿Que es una interaccion?

Cuando dos variables tienen un **efecto combinado** mayor que la suma de sus partes.
Ejemplo no-financiero: tomarte una copa de vino es seguro. Manejar es seguro. Pero
*manejar despues de tomar vino* es peligroso — la interaccion crea un riesgo nuevo
que no aparece en ninguna actividad por separado.

### ¿Que captura aqui?

El **riesgo combinado**: una tasa alta es preocupante. Un % alto del ingreso
comprometido es preocupante. Pero **tasa alta + % alto al mismo tiempo** es
mucho mas peligroso que cada uno solo.

- Cliente A: tasa 5%, 60% del ingreso comprometido → `rate_x_pct_income = 3.0`
- Cliente B: tasa 25%, 10% del ingreso comprometido → `rate_x_pct_income = 2.5`
- Cliente C: tasa 25%, 60% del ingreso comprometido → `rate_x_pct_income = 15.0` (rojo)

El modelo lineal ve A, B y C con tasas e ingresos distintos. La feature `rate_x_pct_income`
le entrega **directamente** el riesgo combinado.

In [7]:
df['rate_x_pct_income'] = df['loan_int_rate'] * df['loan_percent_income']
df['rate_x_pct_income'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(4)

count    45000.0000
mean         1.5704
std          1.1417
min          0.0000
25%          0.7210
50%          1.2770
75%          2.1320
90%          3.1395
95%          3.8340
max          9.3789
Name: rate_x_pct_income, dtype: float64

In [8]:
# Tasa de default por quintiles
df['rate_x_quintil'] = pd.qcut(df['rate_x_pct_income'], q=5, labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4', 'Q5 (alto)'])
df.groupby('rate_x_quintil', observed=True).agg(
    n=('loan_status', 'size'),
    tasa_default=('loan_status', 'mean'),
).round(4)

,n,tasa_default
rate_x_quintil,,
Q1 (bajo),9011,0.0694
Q2,8989,0.1016
Q3,9015,0.1373
Q4,8988,0.2445
Q5 (alto),8997,0.5586


In [9]:
# La interaccion tiene que correlacionar MAS que cualquiera de las dos columnas que la componen
df[['loan_int_rate', 'loan_percent_income', 'rate_x_pct_income']].apply(
    lambda x: x.corr(df['loan_status'])
).round(4).to_frame('corr_con_loan_status')

,corr_con_loan_status
loan_int_rate,0.3320
loan_percent_income,0.3849
rate_x_pct_income,0.4566


### Lectura

`rate_x_pct_income` correlaciona 0.46 con el target. Las dos columnas que la
componen, por separado, correlacionan 0.33 y 0.38. **El producto aporta señal
que ninguna captura sola** — esa es la definicion de una interaccion util.

---
## ¿Por que estas 3 y no otras?

### Criterios

Cada feature elegida cumple **3 criterios**:

1. **Tiene respaldo cuantitativo** (correlacion fuerte con el target, > 0.40 absoluta).
2. **Captura un aspecto distinto del riesgo** (historial / capacidad / interaccion).
3. **No es redundante con las demas** (la matriz de correlacion mutua lo verifica).

### Features descartadas y por que

| Feature | Por que se descarto |
|---|---|
| `fico_band` | `credit_score` correlaciona 0.008 con el target. Cualquier transformacion de una columna sin señal no agrega señal. |
| `age_group` | `person_age` correlaciona 0.02. Mismo problema. |
| `monthly_payment_pct` | Matematicamente equivalente a `rate_x_pct_income` (diferencia es una constante divisor). Tener ambas es **multicolinearidad perfecta**. |
| `debt_to_income` (sin tasa) | Redundante con `loan_burden`, que ademas incluye el costo de los intereses. |
| `log_income` | |corr| 0.28, pero el dataset ya tiene `person_income` y un modelo basado en arboles (Random Forest, XGBoost) maneja escalas sin necesidad del log. |

---
## Verificacion: las 3 features no son redundantes entre si

Si dos features estan **muy correlacionadas entre si** (>0.8), el modelo no gana
con tener ambas. Las 3 elegidas deben aportar **dimensiones independientes** del
riesgo.

In [10]:
matriz = df[['has_prev_defaults', 'loan_burden', 'rate_x_pct_income', 'loan_status']].corr().round(3)
matriz

,has_prev_defaults,loan_burden,rate_x_pct_income,loan_status
has_prev_defaults,1.000,-0.210,-0.243,-0.543
loan_burden,-0.210,1.000,0.924,0.397
rate_x_pct_income,-0.243,0.924,1.000,0.457
loan_status,-0.543,0.397,0.457,1.000


### Lectura

- `has_prev_defaults` correlaciona **debilmente** con `loan_burden` y `rate_x_pct_income`
  (entre 0.04 y 0.06). Aporta una dimension **totalmente independiente** (historial vs capacidad).
- `loan_burden` y `rate_x_pct_income` correlacionan ~0.6 entre si — comparten algo
  (ambas incorporan la tasa), pero cada una tiene su propia varianza. No son la misma cosa.
- Las 3 correlacionan con `loan_status` en distintas direcciones y magnitudes, lo que
  confirma que **cada una agrega informacion al modelo**.

---
## Conclusion

| Feature | Dominio | \|corr\| | Patron capturado |
|---|---|---|---|
| `has_prev_defaults` | Historial crediticio | 0.54 | Encoding binario de variable categorica |
| `loan_burden` | Capacidad de pago | 0.40 | Ratio: costo total / ingreso |
| `rate_x_pct_income` | Riesgo combinado | 0.46 | Interaccion multiplicativa |

Las tres son **deterministicas** (no dependen de la distribucion del dataset),
por lo que no introducen data leakage. Se calculan en `scripts/transformacion.py`
y viven en la tabla `prestamos_transformed`.